In [21]:
# install everything this notebook needs
!pip install pika redis flask -q

In [22]:
# stand up redis and rabbitmq inside this colab session
!apt-get install -y -qq redis-server rabbitmq-server > /dev/null

!redis-server --daemonize yes
!service rabbitmq-server start

 * Starting RabbitMQ Messaging Server rabbitmq-server         * RabbitMQ Messaging Server already running
[ OK ]


In [23]:
import time

# rabbitmq takes a while to fully come up, redis is basically instant
time.sleep(8)

!redis-cli ping
!rabbitmqctl status | head -3

PONG
Status of node rabbit@c976c43d70a2 ...
Runtime



In [24]:
# upload the pretrained model that came out of the training notebook
from google.colab import files
import pickle

print('please upload yield_model_pipeline.pkl')
uploaded_model = files.upload()

model_file = next(iter(uploaded_model))

with open(model_file, 'rb') as f:
    bundle = pickle.load(f)

model = bundle['pipeline']
num_feats = bundle['num_feats']
cat_feats = bundle['cat_feats']
required_fields = num_feats + cat_feats

print('model loaded, expecting these fields on every request:')
print(required_fields)


please upload yield_model_pipeline.pkl


Saving yield_model_pipeline.pkl to yield_model_pipeline (1).pkl
model loaded, expecting these fields on every request:
['urban', 'family_income', 'first_gen', 'parent_grad', 'cutoff_12th', 'entrance_score', 'tuition', 'distance_km', 'competing_offers', 'merit_aid_pct', 'need_aid_pct', 'total_aid_pct', 'aid_amount', 'net_price', 'district', 'category', 'college_tier']


In [25]:
# ---- redis config ----
# same as the phase 4 notebook - replace with the real dev values once available

'''
REDIS_HOST = 'localhost'
REDIS_PORT = 6379
REDIS_PASSWORD = None
REDIS_DB = 0
REDIS_USE_TLS = False
'''

REDIS_HOST = '129.153.75.221'
REDIS_PORT = 6379
REDIS_USERNAME = 'default'
REDIS_PASSWORD = 'cwe5cU6Tyzvd'
REDIS_DB = 0                 # not given, 0 is the standard default db
REDIS_USE_TLS = False        # not stated either way, leaving off unless told otherwise

CACHE_TTL_SECONDS = 3600   # how long a cached prediction stays valid for

In [26]:
# ---- rabbitmq config ----
# same as the phase 3 notebook - replace with the real dev values once available

'''
RABBITMQ_HOST = 'localhost'
RABBITMQ_PORT = 5672
RABBITMQ_USERNAME = 'guest'
RABBITMQ_PASSWORD = 'guest'
RABBITMQ_VHOST = '/'
RABBITMQ_USE_TLS = False
'''

RABBITMQ_HOST = '129.153.75.221'
RABBITMQ_PORT = 5672             # not given, using the standard amqp port since TLS is not required
RABBITMQ_USERNAME = 'bytesmart_interns'
RABBITMQ_PASSWORD = 'YaZU4ghFdBzY'
RABBITMQ_VHOST = '/'
RABBITMQ_USE_TLS = False

QUEUE_NAME = 'yield_prediction_queue'

In [27]:
import redis

redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=REDIS_DB,
    ssl=REDIS_USE_TLS,
    decode_responses=True
)

print('connected to redis:', redis_client.ping())

connected to redis: True


In [28]:
import pika
import ssl

# same connection helper as the phase 3 notebook. kept as a function (not one
# shared connection) because the flask thread and the consumer thread below
# both need their own connection - pika connections are not thread safe.
def get_rabbitmq_connection():

    credentials = pika.PlainCredentials(RABBITMQ_USERNAME, RABBITMQ_PASSWORD)

    if RABBITMQ_USE_TLS:
        ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        ssl_options = pika.SSLOptions(ssl_context, RABBITMQ_HOST)
        connection_params = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials,
            ssl_options=ssl_options
        )
    else:
        connection_params = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials
        )

    return pika.BlockingConnection(connection_params)


# make sure the queue exists before anything tries to publish or consume from it
setup_connection = get_rabbitmq_connection()
setup_channel = setup_connection.channel()
setup_channel.queue_declare(queue=QUEUE_NAME, durable=True)
setup_connection.close()
print('queue ready:', QUEUE_NAME)

queue ready: yield_prediction_queue


In [29]:
import json

# opens a short-lived connection, publishes one message, closes it again.
# simplest safe way to publish from inside a flask request handler without
# fighting over a connection with the background consumer thread below.
def publish_event(message):
    connection = get_rabbitmq_connection()
    channel = connection.channel()
    channel.basic_publish(
        exchange='',
        routing_key=QUEUE_NAME,
        body=json.dumps(message),
        properties=pika.BasicProperties(delivery_mode=2)
    )
    connection.close()

In [30]:
import threading
import datetime

In [31]:
# this is where "process the received message" happens on the consuming side.
# keeping the processed events in a plain list here just so we can prove, in
# this same notebook, that messages published by /predict actually get picked
# up and handled - in a real system this callback would be doing something
# like updating a dashboard, sending a notification, writing to a database etc.
processed_events = []

def handle_prediction_event(ch, method, properties, body):
    event = json.loads(body)
    event['processed_at'] = datetime.datetime.now().isoformat()
    processed_events.append(event)

    print(f"[consumer] processed prediction event for cache key {event['cache_key']} "
          f"-> yield_probability={event['result']['yield_probability']}")

    ch.basic_ack(delivery_tag=method.delivery_tag)

In [32]:
def run_consumer():
    connection = get_rabbitmq_connection()
    channel = connection.channel()
    channel.queue_declare(queue=QUEUE_NAME, durable=True)
    channel.basic_qos(prefetch_count=1)
    channel.basic_consume(queue=QUEUE_NAME, on_message_callback=handle_prediction_event)
    channel.start_consuming()


# runs forever in the background, listening for prediction events
consumer_thread = threading.Thread(target=run_consumer, daemon=True)
consumer_thread.start()

time.sleep(1)
print('consumer thread is running')

consumer thread is running


In [33]:
import hashlib
import pandas as pd
from flask import Flask, request, jsonify

In [34]:
app = Flask(__name__)

NUMERIC_FIELDS = {"urban", "family_income", "first_gen", "parent_grad", "cutoff_12th", "entrance_score",
                   "tuition", "distance_km", "competing_offers", "merit_aid_pct", "need_aid_pct",
                   "total_aid_pct", "aid_amount", "net_price"}


def validate_payload(payload):
    if not isinstance(payload, dict):
        return "request body must be a json object of applicant features"

    missing = [f for f in required_fields if f not in payload]
    if missing:
        return f"missing required fields: {missing}"

    bad_types = [f for f in NUMERIC_FIELDS if f in payload and not isinstance(payload[f], (int, float))]
    if bad_types:
        return f"these fields must be numeric: {bad_types}"

    return None


def make_cache_key(payload):
    # same applicant details should always hash to the same key, so a repeat
    # request for the exact same applicant is a cache hit
    raw = json.dumps(payload, sort_keys=True)
    return 'yield:' + hashlib.md5(raw.encode()).hexdigest()


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'model_loaded': model is not None,
        'redis_connected': redis_client.ping()
    }), 200


@app.route('/predict', methods=['POST'])
def predict():
    payload = request.get_json(silent=True)

    error = validate_payload(payload)
    if error:
        return jsonify({'error': error}), 400

    cache_key = make_cache_key(payload)

    # step 2 - check redis for a cached result first
    cached = redis_client.get(cache_key)
    if cached:
        result = json.loads(cached)
        result['source'] = 'cache'
        return jsonify(result), 200

    try:
        # step 3 - not cached, so invoke the ml model
        row = pd.DataFrame([payload])[required_fields]
        proba = float(model.predict_proba(row)[:, 1][0])
        result = {
            'yield_probability': proba,
            'predicted_enrolled': proba >= 0.5
        }

        # step 4 - store the result in redis for next time
        redis_client.set(cache_key, json.dumps(result), ex=CACHE_TTL_SECONDS)

        # step 5 - publish an event to rabbitmq about this prediction
        event = {
            'event': 'prediction_made',
            'cache_key': cache_key,
            'result': result,
            'timestamp': datetime.datetime.now().isoformat()
        }
        publish_event(event)

        # step 7 - return the result. the consumer (step 6) handles the
        # published event on its own in the background thread above.
        result['source'] = 'model'
        return jsonify(result), 200

    except Exception as e:
        return jsonify({'error': str(e)}), 500


print('flask app defined: GET /health, POST /predict')

flask app defined: GET /health, POST /predict


In [35]:
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)


def is_server_already_running():
    try:
        r = requests.get('http://127.0.0.1:5000/health', timeout=1)
        return r.status_code == 200
    except Exception:
        return False


import requests

if is_server_already_running():
    print('server already running, reusing it')
else:
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
    time.sleep(2)
    print('flask server started on http://127.0.0.1:5000')

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 11:35:25] "GET /health HTTP/1.1" 200 -


server already running, reusing it


In [36]:
# health check first
r = requests.get('http://127.0.0.1:5000/health')
print(r.status_code, r.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 11:35:28] "GET /health HTTP/1.1" 200 -


200 {'model_loaded': True, 'redis_connected': True, 'status': 'ok'}


In [37]:
applicant = {
    'district': 'Madurai', 'category': 'MBC', 'urban': 0, 'family_income': 210000,
    'first_gen': 1, 'parent_grad': 0, 'cutoff_12th': 79.0, 'entrance_score': 128.0,
    'college_tier': 'Tier-2 (Affiliated)', 'tuition': 110000, 'distance_km': 35.0,
    'competing_offers': 1, 'merit_aid_pct': 0.20, 'need_aid_pct': 0.35,
    'total_aid_pct': 0.28, 'aid_amount': 30800, 'net_price': 79200,
}

# first call for this applicant - should be a cache miss, so this goes
# through the model, gets cached in redis, and publishes a rabbitmq event
print('--- call 1, expect source=model ---')
r1 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r1.status_code, r1.json())

# give the background consumer a moment to pick up and process the event
time.sleep(1.5)
print()
print('events processed by the consumer so far:', len(processed_events))
print(processed_events[-1] if processed_events else None)

--- call 1, expect source=model ---


INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 11:35:30] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:07f0dfa67846f10fbb5bed79aadec3dc -> yield_probability=0.3799366710209084
200 {'predicted_enrolled': False, 'source': 'model', 'yield_probability': 0.3799366710209084}

events processed by the consumer so far: 1
{'event': 'prediction_made', 'cache_key': 'yield:07f0dfa67846f10fbb5bed79aadec3dc', 'result': {'yield_probability': 0.3799366710209084, 'predicted_enrolled': False}, 'timestamp': '2026-09-18T11:35:30.412357', 'processed_at': '2026-09-18T11:35:30.649538'}


In [38]:
# second call, same applicant - should now be a cache hit, no model call,
# no new rabbitmq event
print('--- call 2, expect source=cache ---')
r2 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r2.status_code, r2.json())

print()
print('events processed by the consumer (should be unchanged):', len(processed_events))

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 11:35:34] "POST /predict HTTP/1.1" 200 -


--- call 2, expect source=cache ---
200 {'predicted_enrolled': False, 'source': 'cache', 'yield_probability': 0.3799366710209084}

events processed by the consumer (should be unchanged): 1


In [39]:
# manually delete the cached entry, same as the update/delete demo in the
# phase 4 notebook, then call again to confirm it goes back through the
# model and gets re-cached
cache_key = make_cache_key(applicant)
redis_client.delete(cache_key)
print('deleted cache key:', cache_key)

print()
print('--- call 3 after delete, expect source=model again ---')
r3 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r3.status_code, r3.json())

time.sleep(1.5)
print()
print('total events processed by the consumer:', len(processed_events))

deleted cache key: yield:07f0dfa67846f10fbb5bed79aadec3dc

--- call 3 after delete, expect source=model again ---


INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 11:35:37] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:07f0dfa67846f10fbb5bed79aadec3dc -> yield_probability=0.3799366710209084
200 {'predicted_enrolled': False, 'source': 'model', 'yield_probability': 0.3799366710209084}

total events processed by the consumer: 2


In [40]:
# one more check with a validation error - missing fields should come back
# as a 400, not a 500 or a silent wrong prediction
incomplete_applicant = {'district': 'Chennai', 'family_income': 300000}
r4 = requests.post('http://127.0.0.1:5000/predict', json=incomplete_applicant)
print(r4.status_code, r4.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 11:35:43] "POST /predict HTTP/1.1" 400 -


400 {'error': "missing required fields: ['urban', 'first_gen', 'parent_grad', 'cutoff_12th', 'entrance_score', 'tuition', 'distance_km', 'competing_offers', 'merit_aid_pct', 'need_aid_pct', 'total_aid_pct', 'aid_amount', 'net_price', 'category', 'college_tier']"}
